<a href="https://colab.research.google.com/github/B-21-g-23/AquaCropvisb/blob/main/download_GCM_data%2C_missing%2C_biascorrection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORT LIBRARIES
import ee
import geemap
!pip install xee
import xee
import xarray as xr
import pandas as pd

In [ ]:
#Colab: install Earth Engine API
!pip install --upgrade earthengine-api geemap

import ee
import geemap

# Trigger OAuth2 authentication flow
ee.Authenticate()
ee.Initialize(
    project = 'ee-birukgetaneh13',
    opt_url='https://earthengine-highvolume.googleapis.com')


In [ ]:
# Define grid points
grid_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point(39.697, 9.67), {'id': 'G1'}),
    ee.Feature(ee.Geometry.Point(39.45, 9.53), {'id': 'G2'}),
])

# Parameters
models = ['BCC-CSM2-MR']
scenarios = ['ssp585']   # ✅ Future scenario SSP5-8.5
start_year = 2015
end_year = 2090
fallback_scale = 25000  # ~25 km resolution

def extract_daily(image):
    """
    Compute daily relative humidity (%) from specific humidity and temperature.
    """
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

    # Inputs
    q = image.select('huss')       # specific humidity (kg/kg)
    T = image.select('tas')        # temperature (K)
    p = ee.Image.constant(101325)  # assumed surface pressure (Pa)

    # Convert temperature to °C
    T_c = T.subtract(273.15)

    # Mixing ratio r = q / (1 - q)
    r = q.divide(q.multiply(-1).add(1))

    # Actual vapor pressure e = (r * p) / (0.622 + r)
    e = r.multiply(p).divide(r.add(0.622))

    # Convert e from Pa → hPa
    e_hpa = e.divide(100.0)

    # Saturation vapor pressure (Tetens formula, hPa)
    e_s = T_c.expression(
        '6.112 * exp((17.67 * T) / (T + 243.5))',
        {'T': T_c}
    )

    # Relative humidity (%), clipped to 0–100
    RH = e_hpa.divide(e_s).multiply(100).max(0).min(100).rename('rh_percent')

    # Sample at grid points
    return RH.sampleRegions(
        collection=grid_points,
        properties=['id'],
        scale=fallback_scale,
        geometries=False
    ).map(lambda f: f.set('date', date))


# --- Main Processing Loop ---
for model in models:
    for scenario in scenarios:

        full_collection = (
            ee.ImageCollection('NASA/GDDP-CMIP6')
            .filterDate(f'{start_year}-01-01', f'{end_year + 1}-01-01')
            .filter(ee.Filter.eq('model', model))
            .filter(ee.Filter.eq('scenario', scenario))
            .select(['huss', 'tas'])  # ✅ use specific humidity + temperature
        )

        # Extract daily RH
        all_data = full_collection.map(extract_daily).flatten()

        # Add metadata
        all_data = all_data.map(lambda f: f.set({
            'scenario': scenario,
            'model': model
        }))

        # Export to Google Drive
        task = ee.batch.Export.table.toDrive(
            collection=all_data,
            description=f'GDDP_CMIP6_{model}_{scenario}_{2015}_{2090}_RH',
            folder='BCC-RH_ssp85',
            fileNamePrefix=f'daily_{model}_{scenario}_{2015}_{2090}_RH',
            fileFormat='CSV'
        )

        task.start()
        print(f"🚀 Export started: {model} | {scenario} | RH only")


🚀 Export started: BCC-CSM2-MR | ssp585 | RH only


# New Section

**filling missing data**

In [ ]:
uploaded = files.upload()


Saving observed.csv to observed.csv


In [ ]:
import pandas as pd

# Use the exact filename shown after upload
df = pd.read_csv("observed.csv")

# Apply linear interpolation
df_interpolated = df.interpolate(method='linear')

# Save the cleaned dataset
df_interpolated.to_csv("observed_filled.csv", index=False)

# Check missing values
print(df_interpolated.isnull().sum())
# Step 8: Download the cleaned file back to your computer
files.download("observed_filled.csv")

**Bias correction of GCM data**

In [ ]:
uploaded = files.upload()

Saving BCC_SSP4.5.csv to BCC_SSP4.5.csv
Saving BCC_SSP8.5.csv to BCC_SSP8.5.csv
Saving observed_filled.csv to observed_filled (1).csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer

# -------------------------------
# 1. Load datasets (CSV format)
# -------------------------------
# Each CSV should have columns: date, solar, humidity, wind
obs = pd.read_csv("observed_filled.csv", parse_dates=["date"])
ssp245 = pd.read_csv("BCC_SSP4.5.csv", parse_dates=["date"])
ssp585 = pd.read_csv("BCC_SSP8.5.csv", parse_dates=["date"])

# -------------------------------
# 2. Define overlap and projection periods
# -------------------------------
overlap_start, overlap_end = "1995-01-01", "2025-12-31"
proj_start, proj_end = "2026-01-01", "2090-12-31"

def split_periods(df, start, end):
    return df[(df["date"] >= start) & (df["date"] <= end)]

# -------------------------------
# 3. Quantile Mapping function
# -------------------------------
def quantile_mapping(obs_series, gcm_hist_series, gcm_future_series):
    # Drop NA values
    obs_vals = obs_series.dropna().values.reshape(-1,1)
    gcm_hist_vals = gcm_hist_series.dropna().values.reshape(-1,1)
    gcm_future_vals = gcm_future_series.dropna().values.reshape(-1,1)

    # Fit QM on GCM historical distribution
    qt_gcm = QuantileTransformer(n_quantiles=100, output_distribution="uniform")
    qt_gcm.fit(gcm_hist_vals)

    # Transform future GCM values into uniform space
    gcm_future_uniform = qt_gcm.transform(gcm_future_vals)

    # Fit QM on observed distribution
    qt_obs = QuantileTransformer(n_quantiles=100, output_distribution="uniform")
    qt_obs.fit(obs_vals)

    # Map back into observed space
    corrected = qt_obs.inverse_transform(gcm_future_uniform)

    # Return as pandas Series with same index
    return pd.Series(corrected.flatten(), index=gcm_future_series.index)

# -------------------------------
# 4. Apply bias correction
# -------------------------------
variables = ["solar", "humidity", "wind"]
corrected = {}

for var in variables:
    # Overlap period
    obs_var = split_periods(obs, overlap_start, overlap_end)[var]
    gcm_hist_245 = split_periods(ssp245, overlap_start, overlap_end)[var]
    gcm_hist_585 = split_periods(ssp585, overlap_start, overlap_end)[var]

    # Projection period
    gcm_future_245 = split_periods(ssp245, proj_start, proj_end)[var]
    gcm_future_585 = split_periods(ssp585, proj_start, proj_end)[var]

    # Bias correction
    corrected[f"{var}_ssp245"] = quantile_mapping(obs_var, gcm_hist_245, gcm_future_245)
    corrected[f"{var}_ssp585"] = quantile_mapping(obs_var, gcm_hist_585, gcm_future_585)

# -------------------------------
# 5. Save corrected outputs
# -------------------------------
df_corrected = pd.DataFrame({
    "date": split_periods(ssp245, proj_start, proj_end)["date"],  # same dates for both scenarios
    "solar_ssp245": corrected["solar_ssp245"],
    "humidity_ssp245": corrected["humidity_ssp245"],
    "wind_ssp245": corrected["wind_ssp245"],
    "solar_ssp585": corrected["solar_ssp585"],
    "humidity_ssp585": corrected["humidity_ssp585"],
    "wind_ssp585": corrected["wind_ssp585"],
})

df_corrected.to_csv("BiasCorrected_SSP245_SSP585_2026_2090.csv", index=False)

print("✅ Bias correction complete. Corrected CSV saved.")
from google.colab import files
files.download("BiasCorrected_SSP245_SSP585_2026_2090.csv")



In [ ]:
uploaded = files.upload()

Saving MRI_SSP4.5.csv to MRI_SSP4.5.csv
Saving MRI_SSP8.5.csv to MRI_SSP8.5.csv
Saving observed_filled.csv to observed_filled (2).csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer

# -------------------------------
# 1. Load datasets (CSV format)
# -------------------------------
# Each CSV should have columns: date, humidity, solar, wind
obs = pd.read_csv("observed_filled.csv", parse_dates=["date"])
ssp245 = pd.read_csv("MRI_SSP4.5.csv", parse_dates=["date"])
ssp585 = pd.read_csv("MRI_SSP8.5.csv", parse_dates=["date"])

# -------------------------------
# 2. Define overlap and projection periods
# -------------------------------
overlap_start, overlap_end = "1995-01-01", "2025-12-31"
proj_start, proj_end = "2026-01-01", "2090-12-31"

def split_periods(df, start, end):
    return df[(df["date"] >= start) & (df["date"] <= end)]

# -------------------------------
# 3. Quantile Mapping function
# -------------------------------
def quantile_mapping(obs_series, gcm_hist_series, gcm_future_series):
    # Drop NA values
    obs_vals = obs_series.dropna().values.reshape(-1,1)
    gcm_hist_vals = gcm_hist_series.dropna().values.reshape(-1,1)
    gcm_future_vals = gcm_future_series.dropna().values.reshape(-1,1)

    # Fit QM on GCM historical distribution
    qt_gcm = QuantileTransformer(n_quantiles=100, output_distribution="uniform")
    qt_gcm.fit(gcm_hist_vals)

    # Transform future GCM values into uniform space
    gcm_future_uniform = qt_gcm.transform(gcm_future_vals)

    # Fit QM on observed distribution
    qt_obs = QuantileTransformer(n_quantiles=100, output_distribution="uniform")
    qt_obs.fit(obs_vals)

    # Map back into observed space
    corrected = qt_obs.inverse_transform(gcm_future_uniform)

    # Return as pandas Series with same index
    return pd.Series(corrected.flatten(), index=gcm_future_series.index)

# -------------------------------
# 4. Apply bias correction
# -------------------------------
variables = ["humidity", "solar", "wind"]
corrected = {}

for var in variables:
    # Overlap period
    obs_var = split_periods(obs, overlap_start, overlap_end)[var]
    gcm_hist_245 = split_periods(ssp245, overlap_start, overlap_end)[var]
    gcm_hist_585 = split_periods(ssp585, overlap_start, overlap_end)[var]

    # Projection period
    gcm_future_245 = split_periods(ssp245, proj_start, proj_end)[var]
    gcm_future_585 = split_periods(ssp585, proj_start, proj_end)[var]

    # Bias correction
    corrected[f"{var}_ssp245"] = quantile_mapping(obs_var, gcm_hist_245, gcm_future_245)
    corrected[f"{var}_ssp585"] = quantile_mapping(obs_var, gcm_hist_585, gcm_future_585)

# -------------------------------
# 5. Save corrected outputs
# -------------------------------
df_corrected = pd.DataFrame({
    "date": split_periods(ssp245, proj_start, proj_end)["date"],  # same dates for both scenarios
    "humidity_ssp245": corrected["humidity_ssp245"],
    "solar_ssp245": corrected["solar_ssp245"],
    "wind_ssp245": corrected["wind_ssp245"],
    "humidity_ssp585": corrected["humidity_ssp585"],
    "solar_ssp585": corrected["solar_ssp585"],
    "wind_ssp585": corrected["wind_ssp585"],
})

output_filename = "BiasCorrected_MRI-ESM2-0_SSP245_SSP585_2026_2090.csv"
df_corrected.to_csv(output_filename, index=False)

print("✅ Bias correction complete for MRI-ESM2-0. Corrected CSV saved.")
from google.colab import files
files.download(output_filename)


✅ Bias correction complete for MRI-ESM2-0. Corrected CSV saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving observed_filled.csv to observed_filled.csv
Saving Raw-hist-MRI.csv to Raw-hist-MRI.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from google.colab import files
import os

# -------------------------------
# 1. Load datasets (use uploaded filenames)
# -------------------------------
# Rename for consistency if needed
if os.path.exists("observed_filled (1).csv"):
    os.rename("observed_filled (1).csv", "observed_filled.csv")
if os.path.exists("Raw-hist-MRI (1).csv"):
    os.rename("Raw-hist-MRI (1).csv", "Raw-hist-MRI.csv")

obs = pd.read_csv("observed_filled.csv", parse_dates=["date"])
historical = pd.read_csv("Raw-hist-MRI.csv", parse_dates=["date"])

# -------------------------------
# 2. Define historical period
# -------------------------------
hist_start, hist_end = "1995-01-01", "2025-12-31"

def split_periods(df, start, end):
    return df[(df["date"] >= start) & (df["date"] <= end)]

# -------------------------------
# 3. Quantile Mapping function
# -------------------------------
def quantile_mapping(obs_series, gcm_series):
    obs_vals = obs_series.dropna().values.reshape(-1,1)
    gcm_vals = gcm_series.dropna().values.reshape(-1,1)

    qt_gcm = QuantileTransformer(n_quantiles=100, output_distribution="uniform")
    qt_gcm.fit(gcm_vals)
    gcm_uniform = qt_gcm.transform(gcm_vals)

    qt_obs = QuantileTransformer(n_quantiles=100, output_distribution="uniform")
    qt_obs.fit(obs_vals)
    corrected = qt_obs.inverse_transform(gcm_uniform)

    return pd.Series(corrected.flatten(), index=gcm_series.index)

# -------------------------------
# 4. Bias Correction for Historical Data
# -------------------------------
variables = ["humidity", "solar", "wind"]
df_out = pd.DataFrame({"date": split_periods(obs, hist_start, hist_end)["date"]})

for var in variables:
    obs_var = split_periods(obs, hist_start, hist_end)[var]
    gcm_hist = split_periods(historical, hist_start, hist_end)[var]

    corr_hist = quantile_mapping(obs_var, gcm_hist)

    # Add to output dataframe
    df_out[f"obs_{var}"] = obs_var.values
    df_out[f"raw_{var}_historical"] = gcm_hist.values
    df_out[f"corr_{var}_historical"] = corr_hist.values

# Save daily output
df_out.to_csv("BiasCorrected_Historical_AllParams_daily.csv", index=False)
print("✅ Daily bias correction complete. CSV saved.")

# -------------------------------
# 5. Convert to Monthly
# -------------------------------
df_out = df_out.set_index("date")

monthly_df = pd.DataFrame()

# Humidity (mean)
monthly_df["obs_humidity"] = df_out["obs_humidity"].resample("M").mean()
monthly_df["raw_humidity_historical"] = df_out["raw_humidity_historical"].resample("M").mean()
monthly_df["corr_humidity_historical"] = df_out["corr_humidity_historical"].resample("M").mean()

# Solar (sum)
monthly_df["obs_solar"] = df_out["obs_solar"].resample("M").sum()
monthly_df["raw_solar_historical"] = df_out["raw_solar_historical"].resample("M").sum()
monthly_df["corr_solar_historical"] = df_out["corr_solar_historical"].resample("M").sum()

# Wind (mean)
monthly_df["obs_wind"] = df_out["obs_wind"].resample("M").mean()
monthly_df["raw_wind_historical"] = df_out["raw_wind_historical"].resample("M").mean()
monthly_df["corr_wind_historical"] = df_out["corr_wind_historical"].resample("M").mean()

monthly_df = monthly_df.reset_index()
monthly_df.to_csv("BiasCorrected_Historical_AllParams_monthly.csv", index=False)
print("✅ Converted to monthly dataset and saved.")

# -------------------------------
# 6. Plotting and Exporting (Monthly)
# -------------------------------
for var in variables:
    obs_var = monthly_df[f"obs_{var}"]
    raw_var = monthly_df[f"raw_{var}_historical"]
    corr_var = monthly_df[f"corr_{var}_historical"]

    # Time series plot
    plt.figure(figsize=(10,5))
    plt.plot(monthly_df["date"], obs_var, label="Observed", alpha=0.7)
    plt.plot(monthly_df["date"], raw_var, label="Raw Historical", alpha=0.7)
    plt.plot(monthly_df["date"], corr_var, label="Corrected Historical", alpha=0.7)
    plt.title(f"{var.capitalize()} (1995–2025, Monthly Historical)")
    plt.legend()
    plt.savefig(f"{var}_timeseries_historical_monthly.png", dpi=300)
    plt.close()

    # Distribution plot
    plt.figure(figsize=(8,5))
    sns.kdeplot(obs_var, label="Observed", linewidth=2)
    sns.kdeplot(raw_var, label="Raw Historical", linewidth=2)
    sns.kdeplot(corr_var, label="Corrected Historical", linewidth=2)
    plt.title(f"Distribution Comparison: {var.capitalize()} (1995–2025, Monthly Historical)")
    plt.legend()
    plt.savefig(f"{var}_distribution_historical_monthly.png", dpi=300)
    plt.close()

print("✅ All monthly graphs saved as PNG files.")

# -------------------------------
# 7. Zip and Download All Graphs
# -------------------------------
shutil.make_archive("graphs_monthly", "zip", "/content")
files.download("graphs_monthly.zip")


✅ Daily bias correction complete. CSV saved.
✅ Converted to monthly dataset and saved.


/tmp/ipython-input-3959268528.py:76: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_df["obs_humidity"] = df_out["obs_humidity"].resample("M").mean()
/tmp/ipython-input-3959268528.py:77: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_df["raw_humidity_historical"] = df_out["raw_humidity_historical"].resample("M").mean()
/tmp/ipython-input-3959268528.py:78: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_df["corr_humidity_historical"] = df_out["corr_humidity_historical"].resample("M").mean()
/tmp/ipython-input-3959268528.py:81: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_df["obs_solar"] = df_out["obs_solar"].resample("M").sum()
/tmp/ipython-input-3959268528.py:82: FutureWarning: 'M' is deprecated and will be removed in a future version, please

✅ All monthly graphs saved as PNG files.
